In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# ── Load Data ──────────────────────────────────────────────────────────────────
df = pd.read_csv(r'D:\iec\portfolio2\66634-ev-data\Electric_Vehicle_Population_Data.csv')

# Normalise EV type labels
df['EV_Type'] = df['Electric Vehicle Type'].str.strip()
bev = df[df['EV_Type'].str.contains('BEV', na=False)]
phev = df[df['EV_Type'].str.contains('PHEV', na=False)]

total_ev   = len(df)
total_bev  = len(bev)
total_phev = len(phev)
bev_pct    = round(total_bev  / total_ev * 100, 2)
phev_pct   = round(total_phev / total_ev * 100, 2)

# ── Derived Datasets ───────────────────────────────────────────────────────────
# Maker share (exclude 0-count artefacts / NaN makes)
maker_counts = (df['Make'].dropna()
                  .value_counts()
                  .reset_index()
                  .rename(columns={'Make':'Make','count':'Count'}))
maker_counts = maker_counts[maker_counts['Count'] > 0]
maker_counts['Share_Pct'] = (maker_counts['Count'] / total_ev * 100).round(2)
top_maker       = maker_counts.iloc[0]['Make']
top_maker_count = int(maker_counts.iloc[0]['Count'])
top_maker_pct   = maker_counts.iloc[0]['Share_Pct']

# Avg electric range – top 10 makers (exclude 0-range rows)
range_df = df[df['Electric Range'] > 0][['Make','Electric Range']].dropna()
top10_makes = maker_counts['Make'].head(10).tolist()
avg_range = (range_df[range_df['Make'].isin(top10_makes)]
              .groupby('Make')['Electric Range']
              .mean()
              .reset_index()
              .rename(columns={'Electric Range':'Avg_Range'})
              .sort_values('Avg_Range', ascending=False))

# County / City dominance – overall, BEV, PHEV
def top10_area(data, col):
    return (data[col].dropna()
               .value_counts()
               .head(10)
               .reset_index()
               .rename(columns={col:'Area','count':'Count'}))

county_all   = top10_area(df,   'County')
county_bev   = top10_area(bev,  'County')
county_phev  = top10_area(phev, 'County')
city_all     = top10_area(df,   'City')
city_bev     = top10_area(bev,  'City')
city_phev    = top10_area(phev, 'City')

top_county       = county_all.iloc[0]['Area']
top_county_count = int(county_all.iloc[0]['Count'])
top_county_pct   = round(top_county_count / total_ev * 100, 2)
top_city         = city_all.iloc[0]['Area']
top_city_count   = int(city_all.iloc[0]['Count'])
top_city_pct     = round(top_city_count   / total_ev * 100, 2)

# ── Colour palette ─────────────────────────────────────────────────────────────
C_BEV    = '#00D4FF'
C_PHEV   = '#FF6B35'
C_ACCENT = '#7DF9FF'
C_DARK   = '#0A0E1A'
C_CARD   = '#111827'
C_GRID   = 'rgba(255,255,255,0.06)'
C_TEXT   = '#E2E8F0'
C_SUB    = '#94A3B8'

FONT_TITLE = 'Rajdhani'
FONT_BODY  = 'IBM Plex Mono'

google_fonts = (
    "https://fonts.googleapis.com/css2?"
    "family=Rajdhani:wght@400;600;700&"
    "family=IBM+Plex+Mono:wght@400;500&display=swap"
)

# ── Build Figure ───────────────────────────────────────────────────────────────
fig = make_subplots(
    rows=5, cols=3,
    subplot_titles=[
        # row 1 – KPIs (3 merged cells handled via annotations; use blank traces)
        '', '', '',
        # row 2
        'BEV vs PHEV Adoption Ratio',
        'Manufacturer Market Share (All Makers)',
        'Avg Electric Range — Top 10 Makers',
        # row 3
        'Top 10 Counties — BEV',
        'Top 10 Counties — PHEV',
        'County & City KPIs',
        # row 4
        'Top 10 Cities — BEV',
        'Top 10 Cities — PHEV',
        'Maker Share KPIs',
        # row 5
        'Overall Market Mix',
        'EV Type by Top County',
        'EV Type by Top City',
    ],
    specs=[
        [{"type":"domain"}, {"type":"domain"}, {"type":"domain"}],  # row 1 KPI tiles
        [{"type":"xy"},     {"type":"domain"}, {"type":"xy"}],
        [{"type":"xy"},     {"type":"xy"},     {"type":"domain"}],
        [{"type":"xy"},     {"type":"xy"},     {"type":"domain"}],
        [{"type":"domain"}, {"type":"xy"},     {"type":"xy"}],
    ],
    vertical_spacing=0.07,
    horizontal_spacing=0.06,
)

# ══════════════════════════════════════════════════════════════════════════════
# ROW 1 – KPI tiles via Indicator traces
# ══════════════════════════════════════════════════════════════════════════════
def kpi_indicator(val, title, color, row, col):
    fig.add_trace(go.Indicator(
        mode="number",
        value=val,
        title={"text": f"<b>{title}</b>", "font": {"size": 15, "color": C_SUB, "family": FONT_BODY}},
        number={"font": {"size": 38, "color": color, "family": FONT_TITLE},
                "valueformat": ","},
        domain={"x": [0, 1], "y": [0, 1]},
    ), row=row, col=col)

kpi_indicator(total_ev,   f"TOTAL EVs REGISTERED",    C_ACCENT,  1, 1)
kpi_indicator(total_bev,  f"BEV VEHICLES  ({bev_pct}%)",  C_BEV, 1, 2)
kpi_indicator(total_phev, f"PHEV VEHICLES  ({phev_pct}%)", C_PHEV, 1, 3)

# ══════════════════════════════════════════════════════════════════════════════
# ROW 2 – BEV vs PHEV donut | Maker pie | Avg Range bar
# ══════════════════════════════════════════════════════════════════════════════
# 2a – Donut BEV / PHEV
fig.add_trace(go.Pie(
    labels=['BEV','PHEV'],
    values=[total_bev, total_phev],
    hole=0.65,
    marker_colors=[C_BEV, C_PHEV],
    textinfo='label+percent',
    textfont=dict(size=13, family=FONT_BODY, color=C_TEXT),
    hovertemplate='%{label}: %{value:,} vehicles (%{percent})<extra></extra>',
), row=2, col=1)

# 2b – Maker market share donut (top 12 + Others)
top12 = maker_counts.head(12).copy()
others_count = maker_counts.iloc[12:]['Count'].sum()
others_row   = pd.DataFrame({'Make': ['Others'], 'Count': [others_count],
                              'Share_Pct': [round(others_count/total_ev*100,2)]})
pie_data = pd.concat([top12, others_row], ignore_index=True)
colors_pie = px.colors.qualitative.Dark24[:len(pie_data)]

fig.add_trace(go.Pie(
    labels=pie_data['Make'],
    values=pie_data['Count'],
    hole=0.55,
    marker_colors=colors_pie,
    textinfo='label+percent',
    textfont=dict(size=10, family=FONT_BODY, color=C_TEXT),
    hovertemplate='%{label}: %{value:,} (%{percent})<extra></extra>',
    showlegend=False,
), row=2, col=2)

# 2c – Avg range horizontal bars
fig.add_trace(go.Bar(
    x=avg_range['Avg_Range'],
    y=avg_range['Make'],
    orientation='h',
    marker=dict(color=avg_range['Avg_Range'],
                colorscale=[[0,'#003566'],[0.5,C_BEV],[1,C_ACCENT]],
                showscale=False),
    text=avg_range['Avg_Range'].round(1).astype(str) + ' mi',
    textposition='outside',
    textfont=dict(color=C_TEXT, size=10, family=FONT_BODY),
    hovertemplate='%{y}: %{x:.1f} miles<extra></extra>',
), row=2, col=3)

# ══════════════════════════════════════════════════════════════════════════════
# ROW 3 – County BEV | County PHEV | County & City KPI indicators
# ══════════════════════════════════════════════════════════════════════════════
for trace_data, color, r, c in [
    (county_bev,  C_BEV,  3, 1),
    (county_phev, C_PHEV, 3, 2),
]:
    fig.add_trace(go.Bar(
        x=trace_data['Count'],
        y=trace_data['Area'],
        orientation='h',
        marker_color=color,
        opacity=0.85,
        text=trace_data['Count'].apply(lambda v: f'{v:,}'),
        textposition='outside',
        textfont=dict(color=C_TEXT, size=9, family=FONT_BODY),
        hovertemplate='%{y}: %{x:,}<extra></extra>',
    ), row=r, col=c)

# County/City KPI tile
kpi_indicator(top_county_count,
              f"TOP COUNTY: {top_county}  ({top_county_pct}%)",
              C_BEV, 3, 3)

# ══════════════════════════════════════════════════════════════════════════════
# ROW 4 – City BEV | City PHEV | Maker KPI indicator
# ══════════════════════════════════════════════════════════════════════════════
for trace_data, color, r, c in [
    (city_bev,  C_BEV,  4, 1),
    (city_phev, C_PHEV, 4, 2),
]:
    fig.add_trace(go.Bar(
        x=trace_data['Count'],
        y=trace_data['Area'],
        orientation='h',
        marker_color=color,
        opacity=0.85,
        text=trace_data['Count'].apply(lambda v: f'{v:,}'),
        textposition='outside',
        textfont=dict(color=C_TEXT, size=9, family=FONT_BODY),
        hovertemplate='%{y}: %{x:,}<extra></extra>',
    ), row=r, col=c)

# Maker KPI tile
kpi_indicator(top_maker_count,
              f"TOP MAKER: {top_maker}  ({top_maker_pct}%)",
              C_ACCENT, 4, 3)

# ══════════════════════════════════════════════════════════════════════════════
# ROW 5 – Overall mix donut | BEV+PHEV by top county bar | BEV+PHEV top city
# ══════════════════════════════════════════════════════════════════════════════
# 5a – full mix donut with % labels
type_counts = df['EV_Type'].value_counts().reset_index()
type_counts.columns = ['Type', 'Count']
fig.add_trace(go.Pie(
    labels=type_counts['Type'],
    values=type_counts['Count'],
    hole=0.5,
    marker_colors=[C_BEV, C_PHEV],
    textinfo='label+percent+value',
    textfont=dict(size=12, family=FONT_BODY, color=C_TEXT),
    hovertemplate='%{label}: %{value:,}<extra></extra>',
), row=5, col=1)

# 5b – Grouped bar: top county BEV vs PHEV
top_counties = county_all['Area'].head(10).tolist()
bev_cnty  = (bev[bev['County'].isin(top_counties)]['County']
              .value_counts().reindex(top_counties, fill_value=0))
phev_cnty = (phev[phev['County'].isin(top_counties)]['County']
              .value_counts().reindex(top_counties, fill_value=0))

fig.add_trace(go.Bar(name='BEV',  x=top_counties, y=bev_cnty.values,
    marker_color=C_BEV,  opacity=0.85,
    hovertemplate='%{x} BEV: %{y:,}<extra></extra>'), row=5, col=2)
fig.add_trace(go.Bar(name='PHEV', x=top_counties, y=phev_cnty.values,
    marker_color=C_PHEV, opacity=0.85,
    hovertemplate='%{x} PHEV: %{y:,}<extra></extra>'), row=5, col=2)

# 5c – Grouped bar: top city BEV vs PHEV
top_cities = city_all['Area'].head(10).tolist()
bev_city  = (bev[bev['City'].isin(top_cities)]['City']
              .value_counts().reindex(top_cities, fill_value=0))
phev_city = (phev[phev['City'].isin(top_cities)]['City']
              .value_counts().reindex(top_cities, fill_value=0))

fig.add_trace(go.Bar(name='BEV',  x=top_cities, y=bev_city.values,
    marker_color=C_BEV,  opacity=0.85, showlegend=False,
    hovertemplate='%{x} BEV: %{y:,}<extra></extra>'), row=5, col=3)
fig.add_trace(go.Bar(name='PHEV', x=top_cities, y=phev_city.values,
    marker_color=C_PHEV, opacity=0.85, showlegend=False,
    hovertemplate='%{x} PHEV: %{y:,}<extra></extra>'), row=5, col=3)

# ══════════════════════════════════════════════════════════════════════════════
# Global Layout
# ══════════════════════════════════════════════════════════════════════════════
fig.update_layout(
    title=dict(
        text="<b>⚡ EV POPULATION INTELLIGENCE DASHBOARD</b>",
        font=dict(size=28, color=C_ACCENT, family=FONT_TITLE),
        x=0.5, xanchor='center', y=0.99,
    ),
    paper_bgcolor=C_DARK,
    plot_bgcolor=C_CARD,
    font=dict(color=C_TEXT, family=FONT_BODY),
    height=1700,
    margin=dict(t=60, b=40, l=20, r=20),
    barmode='group',
    legend=dict(
        bgcolor='rgba(0,0,0,0)',
        font=dict(color=C_TEXT, size=11),
        orientation='h', x=0.5, xanchor='center', y=-0.01,
    ),
    hoverlabel=dict(bgcolor=C_CARD, font_color=C_TEXT, font_family=FONT_BODY),
)

# Apply dark theme to all xy axes
for axis in [a for a in dir(fig.layout) if a.startswith('xaxis') or a.startswith('yaxis')]:
    fig.layout[axis].update(
        gridcolor=C_GRID,
        linecolor='rgba(255,255,255,0.1)',
        tickfont=dict(color=C_SUB, size=9, family=FONT_BODY),
        title_font=dict(color=C_SUB, size=10, family=FONT_BODY),
        zeroline=False,
    )

# Fix subplot title colours
for ann in fig.layout.annotations:
    ann.font.color  = C_TEXT
    ann.font.family = FONT_TITLE
    ann.font.size   = 13

fig.write_html(
    r'D:\iec\portfolio2\66634-ev-data\ev_dashboard.html',
    include_plotlyjs='cdn',
    config={'displayModeBar': True, 'scrollZoom': True},
    # inject Google Fonts
    post_script=f"",
    full_html=True,
)

print("✅  Dashboard saved → D:\\iec\\portfolio2\\66634-ev-data\\ev_dashboard.html")
print(f"\n{'='*55}")
print(f"  KPI SUMMARY")
print(f"{'='*55}")
print(f"  Total EVs Registered : {total_ev:,}")
print(f"  BEV                  : {total_bev:,}  ({bev_pct}%)")
print(f"  PHEV                 : {total_phev:,}  ({phev_pct}%)")
print(f"  Top Maker            : {top_maker}  ({top_maker_count:,} — {top_maker_pct}%)")
print(f"  Top County           : {top_county}  ({top_county_count:,} — {top_county_pct}%)")
print(f"  Top City             : {top_city}  ({top_city_count:,} — {top_city_pct}%)")
print(f"{'='*55}")